# 01 · Bronze ingestion

Read telemetry JSONL, the fleet SQLite database, and real weather snapshots/API responses. Save their data before Silver cleaning. Fleet and telemetry are synthetic.

**Run order:** Bronze → Silver → Gold → separate Power BI report. Bronze calls the demo preparation notebook in Databricks; you do not need to run setup twice. Reruns overwrite only this demo namespace. All dates use UTC.

## Prepare demo inputs
This generates the fictional source data, prepares the volume and defines the weather helper. It explicitly uses saved real weather responses.

In [ ]:
%run ./00_prepare_demo

## Paths, dates and imports

In [ ]:
from pathlib import Path
from datetime import date
import json
import os
import sqlite3
import uuid

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema_name = "ev_telematics.fleet_activity_demo_20260910"
project_dir = Path("/Volumes/ev_telematics/fleet_activity_demo_20260910/files")
source_dir = project_dir / "data" / "sources"
output_dir = project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
start_date = date(2026, 8, 1)
end_date = date(2026, 8, 30)
run_id = str(uuid.uuid4())

# This notebook uses the Databricks-provided Spark session.
spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Report period:", start_date, "to", end_date)

## Read telemetry into Bronze
Read the timestamp as text so a bad timestamp can be inspected later. PERMISSIVE keeps broken JSON in `_corrupt_record`. Save before querying that column. Expected: 4,316 rows.

In [ ]:
telemetry_schema = StructType([
    StructField("vehicle_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("odometer_km", DoubleType(), True),
    StructField("_corrupt_record", StringType(), True),
])
raw_telemetry = (
    spark.read.json(
        str(source_dir / "telemetry" / "*.jsonl"),
        schema=telemetry_schema,
        mode="PERMISSIVE",
        columnNameOfCorruptRecord="_corrupt_record"
    )
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("run_id", F.lit(run_id))
)
bronze_table = schema_name + ".bronze_telemetry"
raw_telemetry.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(bronze_table)
bronze = spark.table(bronze_table)
bronze.printSchema()
print("Bronze rows:", bronze.count())

## Read the fleet database
Extract the vehicle roster and depot coordinates. Silver will check the roster before Gold uses it.

In [ ]:
database_path = source_dir / "fleet.sqlite"
# Query the driver copy; database_path above is the identical volume copy used for input hashes.
with sqlite3.connect((local_source_dir / "fleet.sqlite").as_uri() + "?mode=ro", uri=True) as database:
    fleet_rows = database.execute("""
        SELECT UPPER(TRIM(v.vehicle_id)), v.model, UPPER(TRIM(v.depot_id)),
               d.depot_name, d.latitude, d.longitude
        FROM vehicles AS v
        LEFT JOIN depots AS d ON v.depot_id = d.depot_id
        ORDER BY v.vehicle_id
    """).fetchall()
    depot_rows = database.execute("SELECT depot_id, latitude, longitude FROM depots ORDER BY depot_id").fetchall()
depots = []
for row in depot_rows:
    depot = {"depot_id": row[0], "latitude": row[1], "longitude": row[2]}
    depots.append(depot)
fleet = spark.createDataFrame(fleet_rows,
    "vehicle_id string, model string, depot_id string, depot_name string, latitude double, longitude double")

fleet.show(4, truncate=False)

## Save fleet Bronze

In [ ]:
fleet.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".bronze_fleet")
print("Fleet rows saved:", fleet.count())

## Ingest weather
The helper saves raw API JSON and validates its response shape. This extracted Bronze table holds the daily fields. Silver checks depot/date coverage and uniqueness. Databricks uses saved real responses; the local notebook also supports live requests.

In [ ]:
# load_weather is defined by 00_prepare_demo.

offline = os.environ.get("FLEET_OFFLINE") == "1"
weather_dir = project_dir / ("examples/weather" if offline else "data/weather")
weather_rows, weather_snapshots = load_weather(
    depots, start_date, end_date, weather_dir,
    refresh=os.environ.get("FLEET_REFRESH_WEATHER") == "1", offline=offline)
weather = spark.createDataFrame(weather_rows,
    "depot_id string, event_date date, temperature_c double, precipitation_mm double")
weather.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".bronze_weather")
print("Weather rows saved:", weather.count())

## Next: 02_silver_cleaning
Bronze tables are saved. Open Silver next. If Bronze fails, stop here and fix it before running downstream notebooks. Old outputs can remain after a failed run.